# Bitcoin Volatility Prediction
### Capstone Final Deliverable

**Name:** `Mohammed Hejairi`
**Topic:** Bitcoin (BTC) Hourly Volatility Prediction
**Cohort:** DSB `2`

---
This notebook is the complete, reproducible technical report: data acquisition (fetched once via CCXT, persisted to CSV, and re-read from disk from then on), cleaning, EDA, feature engineering, feature selection, four tuned models with documented improvement steps, a final model comparison, and a **live out-of-sample testing workflow** to validate predictions against real future data as it arrives.

## Introduction

This project builds an **hourly volatility forecasting pipeline for Bitcoin (BTC/USDT)**, using 10 years of hourly OHLCV history pulled from Binance via [CCXT](https://docs.ccxt.com/). The target is **7-day realized volatility** (rolling standard deviation of hourly returns over a 168-hour window), updated every hour — so the model is always answering "how turbulent has BTC been over the last week, as of right now, and how turbulent will it be next?"

Two design choices set this notebook apart from a typical one-off analysis:
1. **Data is fetched once and saved to CSV.** Every cell after the initial pull reads from that CSV, not from the live API — this makes the notebook fast to re-run, fully reproducible, and gives a fixed historical snapshot to train and evaluate on.
2. **A live testing section at the end** fetches genuinely new candles (tomorrow's, the day after's, etc.) that did not exist when the model was trained, and scores the saved model against them — real out-of-sample validation, not just a held-out slice of historical data.

## Problem Statement + Aim

**The Raw Issue:** Bitcoin's price swings unpredictably at short (hourly) timescales, and traders, exchanges, and risk desks have no reliable way to anticipate how turbulent the next stretch of trading will be — leading to mispriced risk, thin margin buffers, and poorly-timed position sizing.

**The Structured Objective** *(Specific — Business Objective — Measurable):*
We aim to predict Bitcoin's next 7-day realized volatility (rolling standard deviation of hourly returns over a 168-hour window), using lagged price/volume history and technical indicators (RSI, EMA, ATR, Bollinger width) sourced via the CCXT exchange API at hourly resolution. This gives traders and risk desks a continuous, hour-by-hour-updating volatility estimate usable directly in position-sizing and margin decisions, targeting an RMSE that beats both a naive-persistence baseline and a seasonality-aware Holt-Winters baseline.

**Problem Type: Regression.** Real downstream consumers of volatility — Black-Scholes option pricing (σ as a continuous input), Value-at-Risk, and position-sizing formulas — need an actual continuous volatility estimate, not a high/low category.

## Objectives
Key questions guiding the analysis and modeling:
- How does BTC's realized volatility behave over time at hourly resolution — is it clustered (calm stretches punctuated by spikes), and does that clustering persist across many hours (autocorrelation)?
- Does volatility differ systematically by hour-of-day or day-of-week (e.g. US/Asia/Europe session overlaps)?
- What engineered features (lagged returns, rolling volatility, volume changes, technical indicators) are most predictive of near-term volatility?
- How much does hyperparameter tuning improve each model over its untuned baseline?
- Does GARCH's asymmetric (leverage-effect) extension actually improve on plain GARCH(1,1) for BTC, the way it does for equities?
- **Does the best model, trained on 10 years of history, actually generalize to real data collected in the days after training?** (Answered in the Live Testing section, not just estimated from a historical test split.)

**Type of Models:** Regression (continuous `volatility_7d`, computed on hourly data).
**Number of models:** 3 — all genuinely time-series-native: **GARCH-family** (GARCH(1,1) → EGARCH/GJR-GARCH), **HAR-RV** (→ HAR-RV-X), and **LSTM** — each built as a **basic/baseline** version and improved into a **tuned** version. Tabular ML models (Linear Regression, Random Forest, XGBoost) were deliberately excluded: they have no built-in notion of time and only see sequence structure through hand-engineered lag features, which is a meaningfully different category from a model whose equation is time-series-native by construction.
**Evaluation Metrics:** RMSE, MAE, and R², compared against two baselines (naive-persistence, and seasonality-aware Holt-Winters exponential smoothing), using a strict time-based train/test split — plus a live/prospective evaluation on real future data.

# Data Inspection

## Data Dictionary

| Column        | Type    | Description |
| ------------- | ------- | ------- |
| Date          | datetime | timestamp of the hourly candle (crypto trades 24/7, no market close) |
| Open, High, Low, Close | float | OHLC price in USD |
| Volume        | float   | trading volume for the hour |
| hourly_return | float   | engineered: hour-over-hour percent change in Close |
| log_return    | float   | engineered: log(Close_t / Close_t-1), additive over time |
| volatility_7d | float   | engineered: rolling 168-hour (7-day) standard deviation of hourly_return (**target**) |
| ma_7d         | float   | engineered: 168-hour (7-day) moving average of Close |
| volume_chg    | float   | engineered: hour-over-hour percent change in Volume |
| ema_12, ema_26, ema_spread | float | engineered: fast/slow EMA (12/26 hourly periods) and normalized spread — trend strength |
| rsi_14        | float   | engineered: 14-period Relative Strength Index (0-100) |
| atr_14        | float   | engineered: 14-period Average True Range — independent, range-based volatility measure |
| bb_width      | float   | engineered: Bollinger Band width (20-period) — second independent volatility proxy |
| macd_hist     | float   | engineered: MACD histogram — momentum-shift signal |
| hour_of_day   | int     | engineered (Feature Selection only): 0-23, tests for session-time effects on volatility |
| vol_bin       | category| engineered (Feature Selection only): Low/Medium/High tercile of volatility_7d, used for the chi-square test |

**Why CCXT, not yfinance:** `yfinance` is fundamentally an equity-market API — it returns `Dividends`/`Stock Splits` columns that are always 0 for crypto and carry no signal. CCXT pulls exchange-native OHLCV data directly from Binance.

**Why hourly technical-indicator periods (14, 20, 26) are NOT rescaled:** standard practice keeps RSI/ATR/Bollinger period counts as "last N candles" regardless of the underlying bar size — an "RSI-14" on hourly data means the last 14 hours, which is the correct, faster-reacting intraday signal. Only the **target's own window** (`volatility_7d`, `ma_7d`) is explicitly scaled to 168 hourly candles, because its name and real-world meaning ("7-day volatility") must stay fixed regardless of data granularity.

## Data Overview
Source, format, `.head()`, `.info()`, `.describe()`.

**Data pipeline design:** the cell below fetches 10 years of hourly BTC/USDT data via CCXT **once** and writes it to `btc_hourly_ohlcv.csv`. Every cell after this reads from that CSV file, not from the live API. This means: (a) re-running the notebook doesn't require 10 years of API calls again, (b) the historical dataset used for training is a fixed, inspectable file, and (c) the *Live Testing* section near the end can cleanly distinguish "data used to train" (this CSV) from "brand new data fetched afterward" (a second, separate fetch).

In [ ]:
# organize your imports in this cell
import os
import time
import joblib
import numpy as np
import pandas as pd
import ccxt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from arch import arch_model  # GARCH-family volatility models

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

SYMBOL = 'BTC/USDT'
TIMEFRAME = '1h'
SINCE_YEARS = 10
ROLLING_WINDOW = 24 * 7  # 168 hourly candles = 7 days - keeps "7-day volatility" meaning fixed
CSV_PATH = 'data/btc_hourly_ohlcv.csv'

In [ ]:
def fetch_ohlcv_history(exchange, symbol, timeframe='1h', since_ms=None, limit=1000):
    """CCXT returns a max of `limit` candles per call, so we page forward in time
    using the timestamp of the last candle as the next `since` until we reach 'now'."""
    all_candles = []
    since = since_ms
    while True:
        candles = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=limit)
        if not candles:
            break
        all_candles += candles
        since = candles[-1][0] + 1
        if len(candles) < limit:
            break
        time.sleep(exchange.rateLimit / 1000)  # respect exchange rate limits
    return all_candles

exchange = ccxt.binance()

if not os.path.exists(CSV_PATH):
    since_ms = exchange.parse8601(
        (pd.Timestamp.utcnow() - pd.DateOffset(years=SINCE_YEARS)).strftime('%Y-%m-%dT%H:%M:%SZ')
    )
    candles = fetch_ohlcv_history(exchange, SYMBOL, timeframe=TIMEFRAME, since_ms=since_ms)
    df_raw = pd.DataFrame(candles, columns=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
    df_raw['Date'] = pd.to_datetime(df_raw['Timestamp'], unit='ms')
    df_raw = df_raw.drop(columns=['Timestamp']).sort_values('Date').reset_index(drop=True)
    df_raw.to_csv(CSV_PATH, index=False)
    print(f'Fetched {len(df_raw)} candles from the API and saved to {CSV_PATH}')
else:
    print(f'{CSV_PATH} already exists - skipping the API fetch (delete the file to re-fetch).')

In [ ]:
# From here on, EVERYTHING reads from the saved CSV, not the live API.
df_raw = pd.read_csv(CSV_PATH, parse_dates=['Date'])
df_raw = df_raw.sort_values('Date').reset_index(drop=True)

print(f'Shape: {df_raw.shape}')
print(f'Date range: {df_raw["Date"].min()} to {df_raw["Date"].max()}')
print(f'\nColumn dtypes:\n{df_raw.dtypes}')
df_raw.describe()

In [ ]:
df_raw.head()

## Data Cleaning
Handle missing values, outliers, and inconsistencies.

In [ ]:
df = df_raw.copy()

# Engineer returns
df['hourly_return'] = df['Close'].pct_change()
df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
df['volume_chg'] = df['Volume'].pct_change()

# Zero-volume candles (real on thin low-liquidity hours, even for BTC occasionally on some
# exchanges) make pct_change() divide by zero, producing +/-inf, NOT NaN. inf silently survives
# a plain dropna() call, so it's neutralized here BEFORE any rolling calculation uses these
# columns - a rolling std() over a window containing even one inf just produces more inf.
inf_check_cols = ['hourly_return', 'log_return', 'volume_chg']
n_inf = np.isinf(df[inf_check_cols]).sum().sum()
df[inf_check_cols] = df[inf_check_cols].replace([np.inf, -np.inf], np.nan)
print(f'Infinite values found (zero-volume/zero-price candles) and converted to NaN: {n_inf}')

# Now safe to compute the rolling target
df['volatility_7d'] = df['hourly_return'].rolling(ROLLING_WINDOW).std()
df['ma_7d'] = df['Close'].rolling(ROLLING_WINDOW).mean()

print(f'\nMissing values per column:\n{df.isnull().sum()}')

In [ ]:
def compute_indicators(df):
    """Technical indicators, computed manually (not via a library) so every formula is
    transparent. Period counts (12/26/14/20) are kept as raw candle counts, not rescaled for
    hourly data - see the Data Dictionary note on why."""
    close, high, low = df['Close'], df['High'], df['Low']

    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    df['ema_12'] = ema_12
    df['ema_26'] = ema_26
    df['ema_spread'] = (ema_12 - ema_26) / close

    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    df['rsi_14'] = 100 - (100 / (1 + rs))

    prev_close = close.shift(1)
    true_range = pd.concat([
        high - low, (high - prev_close).abs(), (low - prev_close).abs()
    ], axis=1).max(axis=1)
    df['atr_14'] = true_range.rolling(14).mean()

    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    df['bb_width'] = ((bb_mid + 2 * bb_std) - (bb_mid - 2 * bb_std)) / bb_mid

    macd_line = ema_12 - ema_26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    df['macd_hist'] = macd_line - signal_line
    return df

df = compute_indicators(df)
indicator_cols = ['ema_spread', 'rsi_14', 'atr_14', 'bb_width', 'macd_hist']
print('Missing values in new indicator columns:')
print(df[indicator_cols].isnull().sum())

In [ ]:
# The only remaining missing values are structural warm-up periods (up to 26 rows, for the
# 26-period EMA / 20-period Bollinger window) - there's no "before the start of history" to
# impute, so these rows are dropped rather than filled.
feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume', 'hourly_return', 'log_return',
    'volatility_7d', 'ma_7d', 'volume_chg', 'ema_12', 'ema_26', 'ema_spread',
    'rsi_14', 'atr_14', 'bb_width', 'macd_hist'
]
df_clean = df.dropna(subset=feature_cols).reset_index(drop=True)

print(f'Rows with non-positive price: {(df_clean[["Open","High","Low","Close"]] <= 0).any(axis=1).sum()}')

# Extreme-move outliers (flash crashes, etc.) are expected and retained as genuine signal for a
# volatility model - flagging via IQR for awareness, not removal.
q1, q3 = df_clean['hourly_return'].quantile([0.25, 0.75])
iqr = q3 - q1
extreme = ((df_clean['hourly_return'] < q1 - 3 * iqr) | (df_clean['hourly_return'] > q3 + 3 * iqr))
print(f'Extreme hourly-return outliers (retained, IQR x3): {extreme.sum()} of {len(df_clean)} rows')
print(f'Shape after cleaning: {df_clean.shape}')

# Exploratory Data Analysis (EDA)
Trends, relationships, anomalies; univariate, bivariate, and multivariate analysis; correlation matrix.

In [ ]:
# Univariate: target distribution
target_column = 'volatility_7d'
sns.histplot(df_clean[target_column], bins=50, color='#55A868')
plt.title('Distribution of 7-day Rolling Volatility (BTC, hourly)')
plt.xlabel('volatility_7d')
plt.show()
print(df_clean[target_column].describe())

In [ ]:
# Volatility over time
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_clean['Date'], df_clean['volatility_7d'], color='#4C72B0', alpha=0.8, linewidth=0.7)
ax.set_title('BTC 7-day Rolling Volatility Over Time (hourly-updating)')
ax.set_ylabel('volatility_7d')
plt.show()

In [ ]:
# Bivariate: does volatility differ by hour-of-day or day-of-week? (session-time effects)
df_clean['hour_of_day'] = df_clean['Date'].dt.hour
df_clean['day_of_week'] = df_clean['Date'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df_clean, x='hour_of_day', y='volatility_7d', ax=axes[0])
axes[0].set_title('Volatility by Hour of Day (UTC)')
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.boxplot(data=df_clean, x='day_of_week', y='volatility_7d', order=day_order, ax=axes[1])
axes[1].set_title('Volatility by Day of Week')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix for continuous columns
corr_cols = [
    'Close', 'Volume', 'hourly_return', 'log_return', 'volatility_7d', 'ma_7d', 'volume_chg',
    'ema_spread', 'rsi_14', 'atr_14', 'bb_width', 'macd_hist'
]
corr = df_clean[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix - Continuous Features vs. Target')
plt.show()
print(f'Correlation with target ({target_column}):')
print(corr[target_column].sort_values(ascending=False))

## Autocorrelation Check
Before modeling, we check whether `volatility_7d` is autocorrelated (related to its own past values — "volatility clustering"). This justifies using lagged/rolling features and is the statistical property GARCH is built to model.

In [ ]:
series = df_clean['volatility_7d'].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(series, lags=48, ax=axes[0])
axes[0].set_title('ACF - volatility_7d (BTC, hourly)')
plot_pacf(series, lags=48, ax=axes[1])
axes[1].set_title('PACF - volatility_7d (BTC, hourly)')
plt.tight_layout()
plt.show()

adf_result = adfuller(series.dropna())
print(f'ADF Statistic: {adf_result[0]:.4f}, p-value: {adf_result[1]:.4f}')
print('-> Stationary (reject unit root)' if adf_result[1] < 0.05 else '-> Non-stationary (fail to reject unit root)')

## Insights
- `volatility_7d` is right-skewed with fat tails — long calm stretches punctuated by sharp spikes, the classic "volatility clustering" pattern in crypto.
- `atr_14` and `bb_width` correlate strongly with the target, as expected for two independent, range-based volatility measures — cross-checks that the target behaves sensibly and that these features carry real, non-circular signal.
- `rsi_14`, `ema_spread`, and `macd_hist` correlate more weakly with the target directly, consistent with them being *leading* momentum indicators rather than concurrent volatility measures.
- The ACF plot shows autocorrelation persisting across many lags (slow decay, not an immediate drop to zero) — confirms volatility clustering and justifies the lag/rolling features used below.
- Hour-of-day / day-of-week boxplots show whether session-time patterns exist (see plots above) — feeds directly into the chi-square test in Feature Selection.

## EDA for Modeling
- The right-skew in `volatility_7d` motivates testing a log transform of the regression target.
- Confirmed autocorrelation means every feature must be **lagged** (using only information available before the prediction date), never fed in at time *t*.
- `atr_14`/`bb_width` are confirmed methodologically distinct from the target (range-based vs. close-to-close), so they are safe to include as features without target leakage.

# Data Preprocessing & Feature Engineering
Transforming, encoding, and preparing features — implemented here, not just outlined.

In [ ]:
# 1) Lag every feature by 1-3 periods (hours) so the model only ever sees information available
#    BEFORE the prediction time (avoids lookahead leakage). Kept as raw hourly lags (not scaled
#    to a calendar span) since these are meant to capture immediate short-term momentum.
LAGS = [1, 2, 3]
lag_source_cols = [c for c in feature_cols if c != 'Date']

df_model = df_clean.copy()
for col in lag_source_cols:
    for lag in LAGS:
        df_model[f'{col}_lag{lag}'] = df_model[col].shift(lag)

# 2) Log-transform the target for a secondary check on the right-skew seen in EDA
df_model['volatility_7d_log'] = np.log1p(df_model['volatility_7d'])

# 3) Drop rows with NaNs introduced by lagging
df_model = df_model.dropna().reset_index(drop=True)
print(f'Model-ready shape: {df_model.shape}')

In [ ]:
# 4) Feature matrix X (lagged columns only - never same-time columns, which would leak
#    the answer) and target y.
target = 'volatility_7d'
feature_columns = [c for c in df_model.columns if c.endswith(tuple(f'_lag{l}' for l in LAGS))]
X = df_model[feature_columns]
y = df_model[target]
print(f'Features: {len(feature_columns)}   Target: {target}')
X.head()

In [ ]:
# 5) Strict TIME-BASED train/test split (never random shuffle - autocorrelated time-series
#    data, and a random split would leak information from adjacent-in-time rows).
split_idx = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {X_train.shape[0]} rows ({df_model["Date"].iloc[0]} to {df_model["Date"].iloc[split_idx-1]})')
print(f'Test:  {X_test.shape[0]} rows ({df_model["Date"].iloc[split_idx]} to {df_model["Date"].iloc[-1]})')
print('(time-ordered split, not shuffled)')

In [ ]:
# 6) Scaling: StandardScaler fit ONLY on training data, then applied to both train and test.
#    Required for the LSTM (and the HAR-RV-X Ridge model, though its regressors are already
#    on a comparable scale); GARCH fits directly on returns and doesn't use this scaler at all.
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

tscv = TimeSeriesSplit(n_splits=5)

# Feature Selection
## Correlation Matrix (Continuous Columns)
Already computed above in EDA. Features with the strongest linear correlation to the target — `atr_14`, `bb_width`, `log_return`, `volume_chg` and their lags — are the primary signal; weaker-correlation features (`rsi_14`, `ema_spread`, `macd_hist`) are retained anyway since EDA identified them as *leading* indicators whose value shows up more in nonlinear feature importance than in linear correlation.

## Chi-Square Analysis (Categorical Columns)
With a single coin, there's no "coin identity" categorical column left to test. Instead, the natural categorical question for hourly single-asset data is: **does the hour of day carry information about the volatility regime?** (session-overlap effects — US/Asia/Europe trading hours). We bin `volatility_7d` into Low/Medium/High terciles (diagnostic only — never used as a modeling feature or target, since the model itself stays a regressor on the continuous value) and test its association with `hour_of_day` via a chi-square test of independence.

In [ ]:
df_clean['vol_bin'] = pd.qcut(df_clean['volatility_7d'], q=3, labels=['Low', 'Medium', 'High'])

contingency = pd.crosstab(df_clean['hour_of_day'], df_clean['vol_bin'])
chi2, p_value, dof, expected = chi2_contingency(contingency)

print(contingency)
print(f'\nChi-square statistic: {chi2:.2f}')
print(f'Degrees of freedom: {dof}')
print(f'p-value: {p_value:.6f}')
print('-> Hour of day IS significantly associated with volatility regime (p < 0.05)' if p_value < 0.05
      else '-> No significant association found (p >= 0.05)')

# Model Building
Three models, each **genuinely time-series-native** by construction — not generic tabular ML fed lag features. Each is shown as a **basic/baseline** version and an **improved** version, with the improvement step documented. Two baselines are included as sanity floors: naive-persistence, and Holt-Winters exponential smoothing (which additionally tests whether the hour-of-day seasonality found in EDA is exploitable).

**Why not Linear Regression / Random Forest / XGBoost:** those models have no built-in notion of time — they only "see" sequence structure through whatever lag columns are engineered by hand, and are agnostic to whether the rows came from a time series or an unrelated tabular dataset. The three models below are different in kind: GARCH's equation is recursive and self-referential by definition, HAR-RV's regressors *are* volatility measured at different time horizons (not arbitrary features), and LSTM reads the raw ordered sequence directly rather than a flattened row. Every model here is time-series-native by construction, not by feature engineering.

In [ ]:
os.makedirs('models', exist_ok=True)

def evaluate(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'{label:35s}  RMSE={rmse:.6f}  MAE={mae:.6f}  R2={r2:.4f}')
    return {'Model': label, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

results = []

naive_pred = X_test['volatility_7d_lag1']
results.append(evaluate(y_test, naive_pred, 'Naive Persistence (baseline)'))

### Baseline: Holt-Winters Exponential Smoothing
A second baseline, included specifically to test one thing the three main models don't explicitly handle: **is the hour-of-day / day-of-week seasonality found in EDA actually exploitable for forecasting?**

Exponential smoothing forecasts a series as a weighted average of its own past, with weights decaying exponentially into the past. Holt-Winters extends this with explicit trend and *seasonal* components:
$$\hat{y}_{t+h} = \underbrace{\ell_t}_{\text{level}} + \underbrace{h \cdot b_t}_{\text{trend}} + \underbrace{s_{t+h-m}}_{\text{seasonal}}$$

It is included as a **baseline, not a main model**, because it smooths the level of the volatility series rather than modeling volatility's *response to shocks* (which is what GARCH's $\alpha\epsilon_{t-1}^2$ term does) — it is structurally a sophisticated persistence model. If it beats naive-persistence meaningfully, the seasonality is real and worth exploiting; if it doesn't, that's a clean negative result worth reporting.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Seasonal period = 24 hours (daily cycle). Fit on the training volatility series, forecast
# forward across the test horizon.
hw_train = df_model['volatility_7d'].iloc[:split_idx]
hw_test = df_model['volatility_7d'].iloc[split_idx:]

try:
    hw_model = ExponentialSmoothing(
        hw_train, trend='add', seasonal='add', seasonal_periods=24,
        initialization_method='estimated'
    ).fit()
    hw_pred = hw_model.forecast(len(hw_test)).values
    print(f'Holt-Winters fitted. AIC: {hw_model.aic:.2f}')
    print(f'Smoothing level (alpha): {hw_model.params["smoothing_level"]:.4f}')
    print(f'Smoothing trend  (beta):  {hw_model.params["smoothing_trend"]:.4f}')
    print(f'Smoothing seasonal (gamma): {hw_model.params["smoothing_seasonal"]:.4f}\n')
    results.append(evaluate(hw_test.values, hw_pred, 'Holt-Winters (baseline)'))
except Exception as e:
    # Holt-Winters can fail to converge on long, noisy series - report rather than crash the run
    print(f'Holt-Winters failed to fit: {e}')
    print('Skipping this baseline - the three main models below are unaffected.')

In [ ]:
df_clean_dated = df_clean.set_index('Date')
returns_pct = df_clean_dated['hourly_return'].dropna() * 100  # arch expects percentage returns

train_end_date = df_model['Date'].iloc[split_idx - 1]
test_start_date = df_model['Date'].iloc[split_idx]
test_end_date = df_model['Date'].iloc[-1]

returns_train = returns_pct.loc[:train_end_date]
returns_test = returns_pct.loc[test_start_date:test_end_date]

## Model 1: GARCH-Family
**Basic version: GARCH(1,1).** The workhorse econometric volatility model — today's variance = a baseline level + a reaction to yesterday's shock + persistence from yesterday's variance:
$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

**Improvement: EGARCH(1,1) or GJR-GARCH(1,1) — whichever fits better by AIC.** Both extend GARCH to capture the **leverage effect**: crypto (like equities) tends to get *more* volatile after a price drop than after an equally-sized rise, an asymmetry vanilla GARCH structurally cannot represent (it treats a +5% and -5% shock identically). This isn't a bigger hyperparameter grid — it's a genuine change to the model's equation. Model order/distribution is selected via **AIC** (the econometric equivalent of cross-validated model selection: information criteria trade off fit against complexity), scanning a small grid rather than a single guess.

In [ ]:
# Basic: GARCH(1,1), normal errors
garch_basic = arch_model(returns_train, vol='GARCH', p=1, q=1, dist='normal')
garch_basic_fit = garch_basic.fit(disp='off')
print(garch_basic_fit.summary())
print(f'\nAIC: {garch_basic_fit.aic:.2f}')

garch_basic_forecast = garch_basic_fit.forecast(horizon=len(returns_test), reindex=False)
garch_basic_pred = (np.sqrt(garch_basic_forecast.variance.values[-1]) / 100)[:len(y_test)]
results.append(evaluate(y_test.values, garch_basic_pred, 'GARCH(1,1) - Basic'))

In [ ]:
# Improvement: scan EGARCH and GJR-GARCH, each with normal and Student-t errors (fat tails are
# well documented in crypto returns), select the lowest-AIC specification - the econometric
# equivalent of a hyperparameter search, using a principled selection criterion instead of a
# validation-set score (appropriate here since these are fit by maximum likelihood, not a loss
# function scikit-learn's search tools expect).
candidate_specs = [
    {'vol': 'EGARCH', 'p': 1, 'o': 1, 'q': 1, 'dist': 'normal'},
    {'vol': 'EGARCH', 'p': 1, 'o': 1, 'q': 1, 'dist': 't'},
    {'vol': 'GARCH',  'p': 1, 'o': 1, 'q': 1, 'dist': 'normal'},  # GJR-GARCH = GARCH with o=1
    {'vol': 'GARCH',  'p': 1, 'o': 1, 'q': 1, 'dist': 't'},
]

best_aic, best_spec, best_fit = np.inf, None, None
for spec in candidate_specs:
    m = arch_model(returns_train, vol=spec['vol'], p=spec['p'], o=spec['o'], q=spec['q'], dist=spec['dist'])
    fit = m.fit(disp='off')
    print(f"{spec}  ->  AIC={fit.aic:.2f}")
    if fit.aic < best_aic:
        best_aic, best_spec, best_fit = fit.aic, spec, fit

print(f'\nBest specification by AIC: {best_spec}')
print(best_fit.summary())

garch_tuned_forecast = best_fit.forecast(horizon=len(returns_test), reindex=False)
garch_tuned_pred = (np.sqrt(garch_tuned_forecast.variance.values[-1]) / 100)[:len(y_test)]
model_name = f"{'EGARCH' if best_spec['vol']=='EGARCH' else 'GJR-GARCH'}({best_spec['dist']}) - Tuned"
results.append(evaluate(y_test.values, garch_tuned_pred, model_name))

joblib.dump(best_spec, 'models/garch_best_spec.pkl')
print(f"\nSaved GARCH spec for live re-fitting: {best_spec}")

## Model 2: HAR-RV / HAR-RV-X
**Basic version: HAR-RV.** Regresses current volatility on volatility averaged over the last day, week, and month (24h / 168h / 720h at hourly resolution) — a well-established realized-volatility benchmark (Corsi, 2009), and simple to implement as Linear Regression on 3 horizon-based regressors. Note: even though the estimator is "Linear Regression," this is *not* the same category as the dropped Model 1 — the 3 inputs here are volatility measured at different time horizons, not arbitrary tabular features, which is what makes HAR-RV a genuine econometric time-series model rather than generic ML.

**Improvement: HAR-RV-X.** Extends HAR-RV with an exogenous volatility-related regressor — lagged `atr_14` (an independent, range-based volatility measure from EDA) — plus Ridge regularization, tuned via `GridSearchCV` with `TimeSeriesSplit`. Adding a genuinely different volatility measure as a regressor (not just more lags of the same thing) is a standard, real extension in the realized-volatility literature.

In [ ]:
HOURS_PER_DAY = 24
har_df = pd.DataFrame(index=df_model.index)
har_df['RV_day'] = df_model['volatility_7d'].shift(HOURS_PER_DAY)
har_df['RV_week'] = df_model['volatility_7d'].shift(HOURS_PER_DAY).rolling(HOURS_PER_DAY * 7).mean()
har_df['RV_month'] = df_model['volatility_7d'].shift(HOURS_PER_DAY).rolling(HOURS_PER_DAY * 30).mean()
har_df['ATR_X'] = df_model['atr_14'].shift(1)  # the "X" exogenous regressor for HAR-RV-X
har_df['target'] = df_model['volatility_7d']
har_df = har_df.dropna()

har_split = int(len(har_df) * 0.8)
har_train, har_test = har_df.iloc[:har_split], har_df.iloc[har_split:]

# Basic: plain HAR-RV, 3 regressors only
har_basic = LinearRegression()
har_basic.fit(har_train[['RV_day', 'RV_week', 'RV_month']], har_train['target'])
har_basic_pred = har_basic.predict(har_test[['RV_day', 'RV_week', 'RV_month']])
results.append(evaluate(har_test['target'], har_basic_pred, 'HAR-RV - Basic'))

In [ ]:
# Improvement: HAR-RV-X (add the ATR regressor) + Ridge regularization, alpha tuned via
# GridSearchCV with TimeSeriesSplit
har_x_cols = ['RV_day', 'RV_week', 'RV_month', 'ATR_X']
har_tscv = TimeSeriesSplit(n_splits=5)

har_ridge_grid = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]}
har_search = GridSearchCV(Ridge(random_state=RANDOM_STATE), har_ridge_grid,
                           cv=har_tscv, scoring='neg_root_mean_squared_error', n_jobs=-1)
har_search.fit(har_train[har_x_cols], har_train['target'])
print(f'Best alpha: {har_search.best_params_}')

har_x_tuned = har_search.best_estimator_
har_x_pred = har_x_tuned.predict(har_test[har_x_cols])
results.append(evaluate(har_test['target'], har_x_pred, 'HAR-RV-X - Tuned'))
joblib.dump(har_x_tuned, 'models/har_rv_x_tuned.pkl')

har_coef = pd.Series(har_x_tuned.coef_, index=har_x_cols).sort_values(ascending=False)
har_coef.plot(kind='barh', figsize=(6, 4), color='#8172B2')
plt.title('HAR-RV-X (Tuned) - Regressor Coefficients')
plt.gca().invert_yaxis()
plt.show()

## Model 3: LSTM
The one model here that reads an actual time sequence natively, rather than needing lag columns hand-fed to it. Input is a rolling window of the last `SEQ_LEN` hours of (already-lagged, already-scaled) features.
**Basic version:** a single LSTM layer (50 units), default Adam optimizer, no dropout, 30-hour lookback.
**Improvement:** a small manual search over units, dropout rate, and lookback window length (GridSearchCV doesn't wrap Keras models cleanly without extra tooling).

In [ ]:
def create_sequences(X, y, seq_len):
    X_arr, y_arr = X.values, y.values
    Xs, ys = [], []
    for i in range(seq_len, len(X_arr)):
        Xs.append(X_arr[i - seq_len:i])
        ys.append(y_arr[i])
    return np.array(Xs), np.array(ys)

def build_lstm(n_features, seq_len, units=50, dropout=0.0, learning_rate=0.001):
    model = Sequential([
        LSTM(units, input_shape=(seq_len, n_features)),
        Dropout(dropout),
        Dense(1),
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

In [ ]:
SEQ_LEN_BASE = 30
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, SEQ_LEN_BASE)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test, SEQ_LEN_BASE)

lstm_baseline = build_lstm(n_features=X_train_scaled.shape[1], seq_len=SEQ_LEN_BASE)
lstm_baseline.fit(X_train_seq, y_train_seq, epochs=30, batch_size=32, verbose=0,
                   validation_split=0.1, shuffle=False)

lstm_baseline_pred = lstm_baseline.predict(X_test_seq, verbose=0).flatten()
results.append(evaluate(y_test_seq, lstm_baseline_pred, 'LSTM - Baseline'))

In [ ]:
val_split = int(len(X_train_scaled) * 0.85)
X_tr_inner, X_val_inner = X_train_scaled.iloc[:val_split], X_train_scaled.iloc[val_split:]
y_tr_inner, y_val_inner = y_train.iloc[:val_split], y_train.iloc[val_split:]

param_grid = [
    {'units': 32, 'dropout': 0.1, 'seq_len': 12},
    {'units': 50, 'dropout': 0.2, 'seq_len': 24},
    {'units': 64, 'dropout': 0.2, 'seq_len': 48},
    {'units': 64, 'dropout': 0.3, 'seq_len': 72},
]

best_val_rmse, best_params = np.inf, None
for params in param_grid:
    Xtr_s, ytr_s = create_sequences(X_tr_inner, y_tr_inner, params['seq_len'])
    Xval_s, yval_s = create_sequences(X_val_inner, y_val_inner, params['seq_len'])
    m = build_lstm(X_train_scaled.shape[1], params['seq_len'], params['units'], params['dropout'])
    m.fit(Xtr_s, ytr_s, epochs=25, batch_size=32, verbose=0, shuffle=False)
    val_pred = m.predict(Xval_s, verbose=0).flatten()
    val_rmse = np.sqrt(mean_squared_error(yval_s, val_pred))
    print(f'{params}  ->  val RMSE={val_rmse:.6f}')
    if val_rmse < best_val_rmse:
        best_val_rmse, best_params = val_rmse, params

print(f'\nBest config: {best_params}')

In [ ]:
X_train_seq_t, y_train_seq_t = create_sequences(X_train_scaled, y_train, best_params['seq_len'])
X_test_seq_t, y_test_seq_t = create_sequences(X_test_scaled, y_test, best_params['seq_len'])

lstm_tuned = build_lstm(X_train_scaled.shape[1], best_params['seq_len'], best_params['units'], best_params['dropout'])
lstm_tuned.fit(X_train_seq_t, y_train_seq_t, epochs=30, batch_size=32, verbose=0, shuffle=False)

lstm_tuned_pred = lstm_tuned.predict(X_test_seq_t, verbose=0).flatten()
results.append(evaluate(y_test_seq_t, lstm_tuned_pred, 'LSTM - Tuned'))

lstm_tuned.save('models/lstm_tuned.keras')
joblib.dump(best_params, 'models/lstm_tuned_params.pkl')
joblib.dump(scaler, 'models/feature_scaler.pkl')
joblib.dump(feature_columns, 'models/feature_columns.pkl')
print('Saved: lstm_tuned.keras, lstm_tuned_params.pkl, feature_scaler.pkl, feature_columns.pkl')

# Live Testing on Real Unseen Data
Everything so far evaluates on a historical test split — real, but still data the market has already "seen" in some sense. This section fetches **genuinely new candles** (hours that didn't exist when `btc_hourly_ohlcv.csv` was first saved), scores the saved models on them, and logs predictions so they can be checked against reality once enough time has passed.

**How to actually use this section:**
1. Run it today — it fetches whatever new hourly candles exist since the training CSV's last timestamp, generates a prediction for each, and appends them to `predictions_log.csv`.
2. Come back **tomorrow** and **the day after** and re-run just this section (no need to retrain anything) — each run fetches more new candles and logs more predictions.
3. Once a logged prediction's timestamp is more than 168 hours (7 days) in the past, the *actual* `volatility_7d` for that hour becomes computable (it needs a full week of subsequent returns) — the final cell below automatically finds those and computes a real, live RMSE/MAE. Nothing here is simulated: it's the genuine gap between what the model predicted and what actually happened.

In [ ]:
LIVE_CSV_PATH = 'data/btc_hourly_live.csv'
PRED_LOG_PATH = 'data/predictions_log.csv'
MAX_LOOKBACK_NEEDED = ROLLING_WINDOW + max(LAGS) + 26 + 5  # rolling window + lags + longest EMA span + buffer

def fetch_new_candles(exchange, symbol, timeframe, since_ms):
    """Fetch everything from `since_ms` up to now - reuses the same paginated fetch logic
    as the original historical pull."""
    return fetch_ohlcv_history(exchange, symbol, timeframe=timeframe, since_ms=since_ms)

def build_features_for_scoring(raw_df):
    """Mirrors the Data Cleaning -> Technical Indicators -> Preprocessing steps above exactly,
    so live-fetched data gets IDENTICAL feature engineering to what the models were trained on.
    Takes a raw OHLCV dataframe (with enough history for the rolling windows to be valid) and
    returns it with every engineered + lagged feature column added."""
    d = raw_df.copy().sort_values('Date').reset_index(drop=True)
    d['hourly_return'] = d['Close'].pct_change()
    d['log_return'] = np.log(d['Close'] / d['Close'].shift(1))
    d['volume_chg'] = d['Volume'].pct_change()
    d[['hourly_return', 'log_return', 'volume_chg']] = d[['hourly_return', 'log_return', 'volume_chg']].replace([np.inf, -np.inf], np.nan)
    d['volatility_7d'] = d['hourly_return'].rolling(ROLLING_WINDOW).std()
    d['ma_7d'] = d['Close'].rolling(ROLLING_WINDOW).mean()
    d = compute_indicators(d)
    for col in lag_source_cols:
        for lag in LAGS:
            d[f'{col}_lag{lag}'] = d[col].shift(lag)
    return d

In [ ]:
# 1) Figure out where the training CSV left off, and fetch everything after that point.
last_training_date = df_raw['Date'].max()

if os.path.exists(LIVE_CSV_PATH):
    df_live_existing = pd.read_csv(LIVE_CSV_PATH, parse_dates=['Date'])
    fetch_since = df_live_existing['Date'].max()
else:
    df_live_existing = pd.DataFrame()
    fetch_since = last_training_date

since_ms = exchange.parse8601(fetch_since.strftime('%Y-%m-%dT%H:%M:%SZ'))
new_candles = fetch_new_candles(exchange, SYMBOL, TIMEFRAME, since_ms)

if new_candles:
    df_new = pd.DataFrame(new_candles, columns=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
    df_new['Date'] = pd.to_datetime(df_new['Timestamp'], unit='ms')
    df_new = df_new.drop(columns=['Timestamp'])
    df_new = df_new[df_new['Date'] > fetch_since]  # avoid re-adding the boundary candle

    df_live_all = pd.concat([df_live_existing, df_new], ignore_index=True).drop_duplicates(subset='Date').sort_values('Date')
    df_live_all.to_csv(LIVE_CSV_PATH, index=False)
    print(f'Fetched {len(df_new)} new candles. Live CSV now has {len(df_live_all)} rows '
          f'({df_live_all["Date"].min()} to {df_live_all["Date"].max()}).')
else:
    df_live_all = df_live_existing
    print('No new candles yet (run again later - hourly candles only close once per hour).')

In [ ]:
# 2) Build features for (training-tail + all live data) together, so rolling windows/lags at
#    the boundary are computed correctly, then keep only the genuinely NEW rows for scoring.
context_tail = df_raw.tail(MAX_LOOKBACK_NEEDED)
combined_raw = pd.concat([context_tail, df_live_all], ignore_index=True).drop_duplicates(subset='Date').sort_values('Date').reset_index(drop=True)
combined_features = build_features_for_scoring(combined_raw)

live_scored = combined_features[combined_features['Date'] > last_training_date].dropna(subset=feature_columns).reset_index(drop=True)
print(f'{len(live_scored)} new rows have enough history to be scored (feature warm-up already satisfied).')
live_scored[['Date', 'Close', 'volatility_7d']].head()

In [ ]:
# 3) Load saved models and generate predictions for the new rows.
#    GARCH-family: refit the best specification (chosen by AIC during Model Building) on the
#    combined (historical + live) returns series - GARCH forecasts forward from its own fitted
#    return process, so "loading a saved model" here means loading the SPEC, then refitting
#    fresh each time (fast - seconds, not minutes) rather than deserializing a fitted object.
#    HAR-RV-X and LSTM: load the actually-saved fitted model/weights, as usual.
if len(live_scored) == 0:
    print('No new rows with complete feature history yet - run this section again in a bit '
          '(or tomorrow) once more hourly candles have closed.')
    live_predictions = pd.DataFrame(columns=['Date', 'GARCH_Pred', 'HAR_RV_X_Pred', 'LSTM_Pred'])
else:
    # --- GARCH-family: refit best spec on combined returns, forecast forward ---
    best_spec = joblib.load('models/garch_best_spec.pkl')
    combined_returns_pct = combined_features.set_index('Date')['hourly_return'].dropna() * 100
    live_dates = live_scored['Date'].values

    garch_live = arch_model(combined_returns_pct, vol=best_spec['vol'], p=best_spec['p'],
                             o=best_spec['o'], q=best_spec['q'], dist=best_spec['dist'])
    garch_live_fit = garch_live.fit(disp='off')
    garch_live_forecast = garch_live_fit.forecast(horizon=len(live_scored), reindex=False)
    garch_live_pred = np.sqrt(garch_live_forecast.variance.values[-1]) / 100

    # --- HAR-RV-X: build the same 4 regressors from combined_features, apply saved Ridge model ---
    har_x_live = joblib.load('models/har_rv_x_tuned.pkl')
    har_live_df = pd.DataFrame(index=combined_features.index)
    har_live_df['RV_day'] = combined_features['volatility_7d'].shift(HOURS_PER_DAY)
    har_live_df['RV_week'] = combined_features['volatility_7d'].shift(HOURS_PER_DAY).rolling(HOURS_PER_DAY * 7).mean()
    har_live_df['RV_month'] = combined_features['volatility_7d'].shift(HOURS_PER_DAY).rolling(HOURS_PER_DAY * 30).mean()
    har_live_df['ATR_X'] = combined_features['atr_14'].shift(1)
    har_live_df['Date'] = combined_features['Date']
    har_live_scored = har_live_df[har_live_df['Date'].isin(live_dates)].dropna()
    har_x_pred_live = har_x_live.predict(har_live_scored[['RV_day', 'RV_week', 'RV_month', 'ATR_X']])

    live_predictions = pd.DataFrame({'Date': live_scored['Date']})
    live_predictions['GARCH_Pred'] = garch_live_pred[:len(live_predictions)]
    live_predictions = live_predictions.merge(
        pd.DataFrame({'Date': har_live_scored['Date'], 'HAR_RV_X_Pred': har_x_pred_live}), on='Date', how='left'
    )

    # --- LSTM: needs a sequence window, not a single row ---
    loaded_scaler = joblib.load('models/feature_scaler.pkl')
    loaded_feature_columns = joblib.load('models/feature_columns.pkl')
    from tensorflow.keras.models import load_model
    lstm_live = load_model('models/lstm_tuned.keras')
    lstm_params = joblib.load('models/lstm_tuned_params.pkl')

    full_context_scaled = pd.DataFrame(
        loaded_scaler.transform(combined_features.dropna(subset=feature_columns)[loaded_feature_columns]),
        columns=loaded_feature_columns
    )
    if len(full_context_scaled) > lstm_params['seq_len']:
        lstm_seq_all, _ = create_sequences(full_context_scaled, pd.Series(np.zeros(len(full_context_scaled))), lstm_params['seq_len'])
        n_live = len(live_predictions)
        lstm_live_preds = lstm_live.predict(lstm_seq_all[-n_live:], verbose=0).flatten() if n_live > 0 else []
        live_predictions['LSTM_Pred'] = lstm_live_preds if len(lstm_live_preds) == len(live_predictions) else np.nan
    else:
        live_predictions['LSTM_Pred'] = np.nan

print(f'Generated predictions for {len(live_predictions)} new hours.')
live_predictions.tail()

In [ ]:
# 4) Log predictions (append-only, skip timestamps already logged) so a real track record
#    builds up across multiple days of running this section.
if os.path.exists(PRED_LOG_PATH):
    existing_log = pd.read_csv(PRED_LOG_PATH, parse_dates=['Date'])
    new_log_rows = live_predictions[~live_predictions['Date'].isin(existing_log['Date'])]
    full_log = pd.concat([existing_log, new_log_rows], ignore_index=True)
else:
    full_log = live_predictions.copy()

full_log = full_log.sort_values('Date').reset_index(drop=True)
full_log.to_csv(PRED_LOG_PATH, index=False)
print(f'Prediction log now has {len(full_log)} rows total.')

In [ ]:
# 5) For any logged prediction old enough (168+ hours ago) that the REAL volatility_7d is now
#    computable, look it up and compute genuine live RMSE/MAE per model. This is the actual
#    answer to "does this model work on real, unseen future data" - not an estimate.
cutoff = pd.Timestamp.utcnow().replace(tzinfo=None) - pd.Timedelta(hours=ROLLING_WINDOW)
scoreable_log = full_log[full_log['Date'] <= cutoff].copy()

if len(scoreable_log) == 0:
    print('No predictions are old enough yet to compare against real outcomes - check back in a '
          'few days (each prediction needs a full 168 hours to elapse before its actual '
          'volatility_7d exists). Re-run this whole section again then.')
else:
    actuals = combined_features.set_index('Date')['volatility_7d']
    scoreable_log['Actual'] = scoreable_log['Date'].map(actuals)
    scoreable_log = scoreable_log.dropna(subset=['Actual'])

    print(f'{len(scoreable_log)} predictions now have real outcomes to compare against:\n')
    for model_col in ['GARCH_Pred', 'HAR_RV_X_Pred', 'LSTM_Pred']:
        valid = scoreable_log.dropna(subset=[model_col])
        if len(valid) > 0:
            evaluate(valid['Actual'], valid[model_col], f'{model_col} [LIVE, real future data]')
    scoreable_log[['Date', 'Actual', 'GARCH_Pred', 'HAR_RV_X_Pred', 'LSTM_Pred']].tail(10)

# Conclusion

## Model Comparison
All models scored with the same `evaluate()` function, so RMSE/MAE/R² are directly comparable. Built automatically from the `results` list populated throughout — nothing here is hand-typed.

In [ ]:
comparison_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
comparison_df.style.background_gradient(subset=['RMSE', 'MAE'], cmap='RdYlGn_r').format({'RMSE': '{:.6f}', 'MAE': '{:.6f}', 'R2': '{:.4f}'})

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = comparison_df.sort_values('RMSE', ascending=True)
colors = ['#C44E52' if 'Baseline' in m else '#4C72B0' if 'Tuned' in m else '#8C8C8C' for m in plot_df['Model']]
plt.barh(plot_df['Model'], plot_df['RMSE'], color=colors)
plt.xlabel('RMSE (lower is better)')
plt.title('Model Comparison - Historical Test Set RMSE')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
best_row = comparison_df.iloc[0]
print(f"Best model (historical test set): {best_row['Model']}")
print(f"RMSE={best_row['RMSE']:.6f}  MAE={best_row['MAE']:.6f}  R2={best_row['R2']:.4f}")
print(f"\nImprovement over naive-persistence baseline: "
      f"{(1 - best_row['RMSE'] / comparison_df.loc[comparison_df['Model'].str.contains('Naive'), 'RMSE'].values[0]) * 100:.1f}% lower RMSE")
print("\nNote: this is the historical-test-set ranking. Check the Live Testing section's output "
      "(run over several days) for how the same models perform on real, never-before-seen data.")

**Best model and justification:** *(fill in with the specific model name and numbers once the notebook is run — the cell above prints them automatically)*. Beyond raw RMSE, weigh: interpretability (HAR-RV-X's coefficients and GARCH's ω/α/β/leverage parameters are directly, theoretically meaningful; LSTM is not), training/inference cost (GARCH refits in seconds, LSTM takes far longer), and — uniquely enabled by this notebook's design — **how the historical-test ranking compares to the Live Testing section's real-world ranking** once a few days of genuine future predictions have accumulated. A model that wins on the historical split but underperforms live is a meaningfully different finding than one that wins on both.

## Discussion

**Business implications:** A working hourly volatility forecast is directly usable by traders sizing positions, exchanges setting margin requirements, options desks pricing Black-Scholes σ, and portfolio managers timing entries/exits around expected turbulence. Because the target is continuous (not a high/low flag), it plugs directly into existing risk formulas.

**Limitations:**
- Volatility is inherently noisy and partly driven by information (news, regulatory action, exchange-specific events) absent from OHLCV history — there's a hard ceiling on achievable R² regardless of tuning.
- The GARCH forecast uses a single static fit over the test horizon rather than a proper rolling/expanding re-fit, understating what a production GARCH system could achieve.
- Single-asset (BTC only): no cross-asset spillover signal from other coins is used, unlike an earlier multi-coin version of this project.
- LSTM is data-hungry and prone to overfitting a target this noisy; its result should be read alongside that caveat.
- The Live Testing section can only validate predictions once 168 hours have elapsed per prediction — meaningful live validation requires patience across multiple days/weeks of re-running that section, not a single run.

**Future improvements:**
- News/sentiment features via an LLM-scored headline pipeline.
- A proper walk-forward (expanding-window) refit for GARCH.
- Automating the Live Testing section to run on a schedule (e.g. a daily cron job) rather than manual re-execution, building a longer real-world track record automatically.
- Ensembling the 3 tuned models.

## References & Appendix

**Data source:** [CCXT](https://docs.ccxt.com/) unified exchange API, pulling hourly OHLCV data from Binance (BTC/USDT).

**Libraries used:** `pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy.stats`, `statsmodels`, `scikit-learn`, `tensorflow`/`keras`, `arch`, `ccxt`, `joblib`.

**Key methodological references:**
- Corsi, F. (2009). *A Simple Approximate Long-Memory Model of Realized Volatility.* — HAR-RV model.
- Bollerslev, T. (1986). *Generalized Autoregressive Conditional Heteroskedasticity.* — GARCH model.
- Nelson, D. (1991). *Conditional Heteroskedasticity in Asset Returns: A New Approach.* — EGARCH, capturing the leverage effect.
- Glosten, L., Jagannathan, R., Runkle, D. (1993). *On the Relation between the Expected Value and the Volatility of the Nominal Excess Return on Stocks.* — GJR-GARCH.
- Holt, C. (1957) and Winters, P. (1960). — Exponential smoothing with trend and seasonality (Holt-Winters), used here as a seasonality-aware baseline.

**Artifacts produced by this notebook** (for reproducibility / grading):
- `btc_hourly_ohlcv.csv` — the fixed historical training dataset.
- `btc_hourly_live.csv` — accumulating live data fetched after training.
- `predictions_log.csv` — every live prediction made, with real outcomes once available.
- `models/` — saved GARCH spec, HAR-RV-X model, and LSTM (+ scaler), reloadable without retraining.